In [1]:
!pip install groq python-dotenv numpy tqdm datasets


[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from groq import Groq
from dotenv import load_dotenv
from datasets import load_dataset

import os
from tqdm import tqdm
import re
import random
import pprint

from typing import List, Dict, Any

load_dotenv()
random.seed(0)

client = Groq()
gsm8k_dataset = load_dataset("gsm8k", "main")

gsm8k_train = gsm8k_dataset["train"]
gsm8k_test  = gsm8k_dataset["test"]

c:\Users\jkijo\OneDrive\Desktop\YBIGTA\YBIGTA_newbie_assignment\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def generate_response_using_Llama(
        prompt: str,
        model: str = "llama-3.1-8b-instant"
    ):
    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are a helpful assistant that solves math problems."
                },
                {
                    "role": "user", 
                    "content": prompt
                }
            ],
            model=model,
            temperature=0.3, ### 수정해도 됩니다!
            stream=False
        )
        return chat_completion.choices[0].message.content
    
    except Exception as e:
        print(f"API call error: {str(e)}")
        return None

#### 응답 잘 나오는지 확인해보기

In [3]:
response = generate_response_using_Llama(
    prompt="Hello world!",
)
print(response)

Hello. Is there a math problem you need help with?


#### GSM8K 데이터셋 확인해보기

In [4]:
print("[Question]")
for l in gsm8k_test['question'][0].split("."):
    print(l)
print("="*100)
print("[Answer]")
print(gsm8k_test['answer'][0])

[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


#### Util 함수들
- extract_final_answer: LLM의 응답을 parse하여 최종 결과만 추출 (정답과 비교하기 위해)
- run_benchmark_test: 벤치마크 테스트
- save_final_result: 결과물 제출을 위한 함수

In [11]:
### 수정해도 됩니다!


def extract_final_answer(response: str):
    """
    Extract the final numerical answer from the model's response.
    Returns None if no answer can be extracted.
    """
    if not response:
        return None
    
    # Strategy 1: Look for "Answer:" or "Final Answer:"
    pattern1 = r"(?:Final\s+)?Answer:\s*\$?\s*([0-9,]+(?:\.[0-9]+)?)"
    matches1 = re.findall(pattern1, response, re.IGNORECASE)
    if matches1:
        result = matches1[-1].replace(",", "")
        if result:
            return result
    
    # Strategy 2: Look for "####" (GSM8K format)
    pattern2 = r"####\s*([0-9,]+(?:\.[0-9]+)?)"
    matches2 = re.findall(pattern2, response)
    if matches2:
        result = matches2[-1].replace(",", "")
        if result:
            return result
    
    # Strategy 3: Look for numbers with units
    pattern3 = r"([0-9,]+(?:\.[0-9]+)?)\s*(?:dollars?|meters?|cups?|miles?|minutes?|hours?|days?|years?|items?|eggs?|cents?)"
    matches3 = re.findall(pattern3, response, re.IGNORECASE)
    if matches3:
        result = matches3[-1].replace(",", "")
        if result:
            return result
    
    # Strategy 4: Look for "= NUMBER" or "is NUMBER"
    pattern4 = r"(?:=|is)\s*\$?\s*([0-9,]+(?:\.[0-9]+)?)"
    matches4 = re.findall(pattern4, response, re.IGNORECASE)
    if matches4:
        result = matches4[-1].replace(",", "")
        if result:
            return result
    
    # Strategy 5: Get the last number in the response
    pattern5 = r"([0-9,]+(?:\.[0-9]+)?)"
    matches5 = re.findall(pattern5, response)
    if matches5:
        result = matches5[-1].replace(",", "")
        if result:
            return result
    
    return None






In [16]:
### 수정해도 됩니다!
"""def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    
    Run benchmark test with improved error handling.
    
    correct = 0
    total = 0
    results = []
    failed_extractions = 0

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            
            predicted_answer = extract_final_answer(response)
            
            if predicted_answer is None or predicted_answer == '':
                failed_extractions += 1
                if VERBOSE:
                    print(f"Warning: Could not extract answer for question {i+1}")
                
                results.append({
                    'question': question,
                    'correct_answer': correct_answer,
                    'predicted_answer': None,
                    'response': response,
                    'correct': False
                })
                total += 1
                continue
            
            try:
                cleaned = predicted_answer.replace(",", "").replace(" ", "")
                if cleaned:
                    predicted_answer = float(cleaned)
                else:
                    predicted_answer = None
            except (ValueError, AttributeError) as e:
                if VERBOSE:
                    print(f"Error converting '{predicted_answer}' to float: {e}")
                predicted_answer = None
            
            if predicted_answer is not None:
                diff = abs(predicted_answer - correct_answer)
                is_correct = diff < 1e-5
            else:
                is_correct = False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")
                if failed_extractions > 0:
                    print(f"Failed extractions: {failed_extractions}")

    return results, correct/total if total > 0 else 0"""
### 수정해도 됩니다!
def run_benchmark_test(
        dataset,
        prompt: str,
        model: str = "llama-3.1-8b-instant",
        num_samples: int = 50,
        VERBOSE: bool = False
    ):
    correct = 0
    total   = 0
    results = []

    for i in tqdm(range(min(num_samples, len(dataset)))):
        question = dataset[i]["question"]
        correct_answer = float(re.findall(r'\d+(?:\.\d+)?', dataset[i]["answer"].split('####')[-1])[0])

        response = generate_response_using_Llama(
            prompt=prompt.format(question=question),
            model=model
        )

        if response:
            if VERBOSE:
                print("="*50)
                print(response)
                print("="*50)
            predicted_answer = extract_final_answer(response)

            if isinstance(predicted_answer, str):
                predicted_answer = float(predicted_answer.replace(",", ""))
            
            diff = abs(predicted_answer - correct_answer)
            is_correct = diff < 1e-5 if predicted_answer is not None else False
            
            if is_correct:
                correct += 1
            total += 1
            
            results.append({
                'question': question,
                'correct_answer': correct_answer,
                'predicted_answer': predicted_answer,
                'response': response,
                'correct': is_correct
            })

            if (i + 1) % 5 == 0:
                current_acc = correct/total if total > 0 else 0
                print(f"Progress: [{i+1}/{num_samples}]")
                print(f"Current Acc.: [{current_acc:.2%}]")

    return results, correct/total if total > 0 else 0

In [13]:
def save_final_result(results: List[Dict[str, Any]], accuracy: float, filename: str) -> None:
    result_str = f"====== ACCURACY: {accuracy} ======\n\n"
    result_str += f"[Details]\n"
    
    for idx, result in enumerate(results):
        result_str += f"Question {idx+1}: {result['question']}\n"
        result_str += f"Correct Answer: {result['correct_answer']}\n"
        result_str += f"Predicted Answer: {result['predicted_answer']}\n"
        result_str += f"Correct: {result['correct']}\n\n"
    
    with open(filename, "w", encoding="utf-8") as f:
        f.write(result_str)

#### Direct prompting with few-shot example

In [14]:
def construct_direct_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = "Instruction:\nSolve the following mathematical question and generate ONLY the answer after a tag, 'Answer:' without any rationale.\n"

    for i in range(num_examples):
        cur_question = train_dataset['question'][i]
        cur_answer = train_dataset['answer'][i].split("####")[-1].strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{cur_question}\n"
        prompt += f"Answer:{cur_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt

In [9]:
### 어떤 방식으로 저장되는지 확인해보세요!
PROMPT = construct_direct_prompt(3)
VERBOSE = False

results, accuracy = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT,
    VERBOSE=VERBOSE,
    num_samples=10
)
save_final_result(results, accuracy, "example.txt")

 50%|█████     | 5/10 [00:04<00:06,  1.32s/it]

Progress: [5/10]
Current Acc.: [60.00%]


100%|██████████| 10/10 [00:08<00:00,  1.18it/s]

Progress: [10/10]
Current Acc.: [60.00%]


In [17]:
# TODO: 0 shot, 3 shot, 5 shot direct prompting을 통해 벤치마크 테스트를 한 후, 각각 direct_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 direct_prompting_5.txt
# 항상 num_samples=50 입니다!


# 0-shot Direct Prompting
prompt_0 = construct_direct_prompt(0)
results_0, accuracy_0 = run_benchmark_test(dataset=gsm8k_test, prompt=prompt_0, num_samples=50)
save_final_result(results_0, accuracy_0, "direct_prompting_0.txt")

# 3-shot Direct Prompting  
prompt_3 = construct_direct_prompt(3)
results_3, accuracy_3 = run_benchmark_test(dataset=gsm8k_test, prompt=prompt_3, num_samples=50)
save_final_result(results_3, accuracy_3, "direct_prompting_3.txt")

# 5-shot Direct Prompting
prompt_5 = construct_direct_prompt(5)
results_5, accuracy_5 = run_benchmark_test(dataset=gsm8k_test, prompt=prompt_5, num_samples=50)
save_final_result(results_5, accuracy_5, "direct_prompting_5.txt")

 10%|█         | 5/50 [00:01<00:17,  2.64it/s]

Progress: [5/50]
Current Acc.: [60.00%]


 20%|██        | 10/50 [00:03<00:13,  2.91it/s]

Progress: [10/50]
Current Acc.: [60.00%]


 30%|███       | 15/50 [00:05<00:14,  2.39it/s]

...

Progress: [40/50]
Current Acc.: [75.00%]


 90%|█████████ | 45/50 [03:46<00:23,  4.65s/it]

Progress: [45/50]
Current Acc.: [75.56%]


100%|██████████| 50/50 [04:11<00:00,  5.03s/it]

Progress: [50/50]
Current Acc.: [78.00%]


### Chain-of-Thought prompting with few-shot example
```text
[Question]
Janet’s ducks lay 16 eggs per day
 She eats three for breakfast every morning and bakes muffins for her friends every day with four
 She sells the remainder at the farmers' market daily for $2 per fresh duck egg
 How much in dollars does she make every day at the farmers' market?
====================================================================================================
[Answer]
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
```

[Answer] 아래의 정답을 도출하는 과정을 예시로 달아주면 CoT의 few shot이 되겠죠?

In [18]:
def construct_CoT_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train

    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    # TODO: 프롬프트를 작성해주세요!
    prompt = (
        "Instruction:\n"
        "Solve the following mathematical question step by step.\n"
        "Show your reasoning clearly, then give the final answer in the format:\n"
        "#### <number>\n"
    )

    for i in range(num_examples):
        # TODO: CoT example을 만들어주세요!
        idx = sampled_indices[i]
        question = train_dataset['question'][idx]
        full_answer = train_dataset['answer'][idx]

        
        reasoning, final_answer = full_answer.split("####")
        final_answer = final_answer.strip()

        prompt += f"\n[Example {i+1}]\n"
        prompt += f"Question:\n{question}\n"
        prompt += "Let's think step by step.\n"
        prompt += reasoning.strip() + "\n"
        prompt += f"#### {final_answer}\n"

    prompt += "\nQuestion:\n{question}\nAnswer:"

    return prompt


In [19]:
# TODO: 0 shot, 3 shot, 5 shot CoT prompting을 통해 벤치마크 테스트를 한 후, 각각 CoT_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 CoT_prompting_5.txt
# 항상 num_samples=50 입니다!


cot_prompt_0 = construct_CoT_prompt(0)
cot_results_0, cot_accuracy_0 = run_benchmark_test(dataset=gsm8k_test, prompt=cot_prompt_0, num_samples=50)
save_final_result(cot_results_0, cot_accuracy_0, "CoT_prompting_0.txt")

cot_prompt_3 = construct_CoT_prompt(3)
cot_results_3, cot_accuracy_3 = run_benchmark_test(dataset=gsm8k_test, prompt=cot_prompt_3, num_samples=50)
save_final_result(cot_results_3, cot_accuracy_3, "CoT_prompting_3.txt")

cot_prompt_5 = construct_CoT_prompt(5)
cot_results_5, cot_accuracy_5 = run_benchmark_test(dataset=gsm8k_test, prompt=cot_prompt_5, num_samples=50)
save_final_result(cot_results_5, cot_accuracy_5, "CoT_prompting_5.txt")

 10%|█         | 5/50 [00:02<00:22,  1.97it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:05<00:20,  1.95it/s]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:07<00:18,  1.89it/s]

...

Progress: [40/50]
Current Acc.: [82.50%]


 90%|█████████ | 45/50 [06:41<00:41,  8.24s/it]

Progress: [45/50]
Current Acc.: [82.22%]


100%|██████████| 50/50 [07:32<00:00,  9.04s/it]

Progress: [50/50]
Current Acc.: [82.00%]


### Construct your prompt!!

목표: 본인만의 프롬프트를 통해 정답률을 더 끌어올려보기!
- gsm8k의 train 데이터셋에서 예시를 가져온 다음 (자유롭게!)
- 그 예시들에 대한 풀이 과정을 만들어주세요!
- 모든 것들이 자유입니다! Direct Prompting, CoT Prompting을 한 결과보다 정답률만 높으면 돼요.

In [23]:
### 자유롭게 수정해도 됩니다! 완전히 새로 함수를 만들어도 돼요.
def construct_my_prompt(num_examples: int = 3) -> str:
    train_dataset = gsm8k_train
    sampled_indices = random.sample(
        [i for i in range(len(train_dataset['question']))],
        num_examples
    )

    prompt = (
        "You are a careful mathematician solving word problems.\n"
        "For each question, follow this exact format:\n\n"
        "Step 1: Restate the problem with key quantities.\n"
        "Step 2: Perform calculations step by step.\n"
        "Step 3: Check whether the answer satisfies the question.\n"
        "Final Answer: Write the final numeric answer after '####'.\n\n"
    )

    for i, idx in enumerate(sampled_indices):
        q = train_dataset['question'][idx]
        a = train_dataset['answer'][idx]

        prompt += f"[Example {i+1}]\n"
        prompt += f"Question:\n{q}\n"
        prompt += f"Answer:\n{a}\n\n"

    prompt += (
        "Now solve the following problem using the same format.\n"
        "Question:\n{question}\n"
        "Answer:\n"
    )

    return prompt



In [24]:
# TODO: 만든 0 shot, 3 shot, 5 shot example과 프롬프트를 통해 벤치마크 테스트를 한 후, 각각 My_prompting_{shot: int}.txt로 저장해주세요!
# 예시: shot이 5인 경우 My_prompting_5.txt
# 항상 num_samples=50 입니다!



PROMPT_MY_0 = construct_my_prompt(0)
results_my_0, accuracy_my_0 = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT_MY_0,
    VERBOSE=False,
    num_samples=50
)
save_final_result(results_my_0, accuracy_my_0, "My_prompting_0.txt")


PROMPT_MY_3 = construct_my_prompt(3)
results_my_3, accuracy_my_3 = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT_MY_3,
    VERBOSE=False,
    num_samples=50
)
save_final_result(results_my_3, accuracy_my_3, "My_prompting_3.txt")


PROMPT_MY_5 = construct_my_prompt(5)
results_my_5, accuracy_my_5 = run_benchmark_test(
    dataset=gsm8k_test,
    prompt=PROMPT_MY_5,
    VERBOSE=False,
    num_samples=50
)
save_final_result(results_my_5, accuracy_my_5, "My_prompting_5.txt")

 10%|█         | 5/50 [00:02<00:24,  1.83it/s]

Progress: [5/50]
Current Acc.: [80.00%]


 20%|██        | 10/50 [00:05<00:21,  1.85it/s]

Progress: [10/50]
Current Acc.: [70.00%]


 30%|███       | 15/50 [00:08<00:19,  1.75it/s]

...

Progress: [40/50]
Current Acc.: [85.00%]


 90%|█████████ | 45/50 [07:42<00:47,  9.50s/it]

Progress: [45/50]
Current Acc.: [84.44%]


100%|██████████| 50/50 [08:32<00:00, 10.24s/it]

Progress: [50/50]
Current Acc.: [86.00%]


### 보고서 작성하기
#### 아래의 내용이 포함되면 됩니다!

1. Direct Prompting, CoT Prompting, My Prompting을 0 shot, 3 shot, 5 shot 정답률을 표로 보여주세요!
2. CoT Prompting이 Direct Prompting에 비해 왜 좋을 수 있는지에 대해서 서술해주세요!
3. 본인이 작성한 프롬프트 기법이 CoT에 비해서 왜 더 좋을 수 있는지에 대해서 설명해주세요!
4. 최종적으로, `PROMPTING.md`에 보고서를 작성해주세요!